# Evaluate Model on Validation Set (Tuned Data)

Manchester.Ai_IBM Z Datathon 2026 Project. Runs on IBM LinuxONE.

**Goal:** Establish the baseline performance of our pipeline on the validation data (September 7–8). 

Because this dataset was used to determine the optimal XGBoost early-stopping point and set the 1% alert threshold, evaluating the model here shows its maximum "tuned" potential. We will compare these results against the unseen test set to measure overfitting.

In [1]:
import os, json
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import average_precision_score

SHARED = os.path.expanduser('~/shared')

MODELS = f'{SHARED}/models'

## 1. Load Data and Filter for Validation Split
We load the pre-processed point-in-time features and strictly isolate transactions from the validation period.

In [2]:
# 1. Load the pre-processed features
print("Loading historical feature data...")
val_data = pd.read_pickle(f'{SHARED}/feat_val.pkl')

print(f"Isolated {len(val_data):,} transactions from the validation period (Days 7-8).")


Loading historical feature data...
Isolated 944,726 transactions from the validation period (Days 7-8).


## 2. Load the model and score every payment
We load the XGBoost model and feature list saved by notebook 03 and score every payment in this split.

In [3]:
# 2. Load the model saved by notebook 03
with open(f'{MODELS}/config.json', 'r') as f:
    config = json.load(f)
FEATURES = config['features']

booster = xgb.Booster()
booster.load_model(f'{MODELS}/mule_xgb.json')

# 3. Score every payment
print("Running inference...")
val_data['xgb_score'] = booster.predict(xgb.DMatrix(val_data[FEATURES]))

# Flag the riskiest 1% of payments in this split (same definition as notebook 03)
k = int(len(val_data) * 0.01)
top = np.argsort(-val_data['xgb_score'].values, kind='stable')[:k]
val_data['is_flagged'] = 0
val_data.iloc[top, val_data.columns.get_loc('is_flagged')] = 1

Running inference...


## 3. Calculate Detection Metrics
We evaluate the model using Precision-Recall AUC (PR-AUC) to account for the extreme class imbalance, and Recall at a 1% alert rate to simulate a bank's operational capacity constraint.

In [4]:
# 4. Calculate Metrics
y_true = val_data['y'].values
preds = val_data['xgb_score'].values
flagged = val_data['is_flagged'].values

total_laundering = int(y_true.sum())
caught_laundering = int((y_true * flagged).sum())

pr_auc = average_precision_score(y_true, preds)
recall_at_1_percent = caught_laundering / total_laundering

print("\n=== Validation Set Evaluation (Tuned Data) ===")
print(f"Total Transactions Evaluated : {len(val_data):,}")
print(f"Actual Laundering Events   : {total_laundering:,}")
print(f"Laundering Events Caught   : {caught_laundering:,}")
print("-" * 46)
print(f"Recall @ 1% alert rate     : {recall_at_1_percent:.2%}")
print(f"PR-AUC Score               : {pr_auc:.4f}")


=== Validation Set Evaluation (Tuned Data) ===
Total Transactions Evaluated : 944,726
Actual Laundering Events   : 1,036
Laundering Events Caught   : 807
----------------------------------------------
Recall @ 1% alert rate     : 77.90%
PR-AUC Score               : 0.4524
